<a href="https://colab.research.google.com/github/flahbocchino/cardioia-fase1-tabagismo-vape/blob/main/cardioia_fase1_textos_pdf_para_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
cd /content
rm -rf cardioia-fase1-tabagismo-vape

git clone https://github.com/flahbocchino/cardioia-fase1-tabagismo-vape.git
ls -lh cardioia-fase1-tabagismo-vape/docs/textos


total 14M
-rw-r--r-- 1 root root 204K Feb 12 12:46 bse-3-2-3.pdf
-rw-r--r-- 1 root root 8.1M Feb 12 12:46 caderno_atencao_primaria_29_rastreamento.pdf
-rw-r--r-- 1 root root 142K Feb 12 12:46 CARDIO IA - FLAVIA BOCCHINO.pdf
-rw-r--r-- 1 root root 5.0M Feb 12 12:46 cartilha1-parar-de-fumar.pdf


Cloning into 'cardioia-fase1-tabagismo-vape'...


In [2]:
from pathlib import Path
import re

repo = Path("/content/cardioia-fase1-tabagismo-vape")
in_dir = repo / "docs" / "textos"
out_dir = repo / "docs" / "textos_txt"
out_dir.mkdir(parents=True, exist_ok=True)

def safe_name(name: str) -> str:
    name = name.strip().replace(" ", "_")
    name = re.sub(r"[^A-Za-z0-9._-]+", "", name)
    name = re.sub(r"_+", "_", name)
    return name

# Leitura principal (pypdf) + fallback (pdfplumber)
def extract_text_from_pdf(pdf_path: Path) -> str:
    text = ""

    # 1) pypdf (geralmente suficiente)
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        parts = []
        for i, page in enumerate(reader.pages):
            t = page.extract_text() or ""
            parts.append(t)
        text = "\n\n".join(parts).strip()
    except Exception:
        text = ""

    # 2) fallback: pdfplumber (às vezes pega melhor)
    if len(text) < 200:
        try:
            import pdfplumber
            parts = []
            with pdfplumber.open(str(pdf_path)) as pdf:
                for page in pdf.pages:
                    t = page.extract_text() or ""
                    parts.append(t)
            text2 = "\n\n".join(parts).strip()
            if len(text2) > len(text):
                text = text2
        except Exception:
            pass

    return text.strip()

pdfs = sorted(in_dir.glob("*.pdf"))
print(f"Encontrados {len(pdfs)} PDFs em {in_dir}\n")

results = []
for pdf in pdfs:
    txt = extract_text_from_pdf(pdf)

    out_name = safe_name(pdf.stem) + ".txt"
    out_path = out_dir / out_name

    if len(txt) < 200:
        # Provável PDF escaneado (imagem). Sem OCR aqui.
        out_path.write_text(
            f"[AVISO] Não foi possível extrair texto suficiente deste PDF.\n"
            f"Arquivo: {pdf.name}\n"
            f"Possível motivo: PDF escaneado/imagem.\n",
            encoding="utf-8"
        )
        status = "⚠️ pouco texto (possível escaneado)"
    else:
        out_path.write_text(txt, encoding="utf-8")
        status = "✅ ok"

    results.append((pdf.name, out_path.name, status))

print("Conversão concluída. Arquivos gerados em:", out_dir)
for a, b, s in results:
    print(f"- {a} -> {b}  ({s})")


Encontrados 4 PDFs em /content/cardioia-fase1-tabagismo-vape/docs/textos

Conversão concluída. Arquivos gerados em: /content/cardioia-fase1-tabagismo-vape/docs/textos_txt
- CARDIO IA - FLAVIA BOCCHINO.pdf -> CARDIO_IA_-_FLAVIA_BOCCHINO.txt  (⚠️ pouco texto (possível escaneado))
- bse-3-2-3.pdf -> bse-3-2-3.txt  (⚠️ pouco texto (possível escaneado))
- caderno_atencao_primaria_29_rastreamento.pdf -> caderno_atencao_primaria_29_rastreamento.txt  (⚠️ pouco texto (possível escaneado))
- cartilha1-parar-de-fumar.pdf -> cartilha1-parar-de-fumar.txt  (⚠️ pouco texto (possível escaneado))


In [3]:
%%bash
ls -lh /content/cardioia-fase1-tabagismo-vape/docs/textos_txt


total 16K
-rw-r--r-- 1 root root 126 Feb 12 12:49 bse-3-2-3.txt
-rw-r--r-- 1 root root 157 Feb 12 12:49 caderno_atencao_primaria_29_rastreamento.txt
-rw-r--r-- 1 root root 144 Feb 12 12:49 CARDIO_IA_-_FLAVIA_BOCCHINO.txt
-rw-r--r-- 1 root root 141 Feb 12 12:49 cartilha1-parar-de-fumar.txt


In [4]:
%%bash
apt-get update -qq
apt-get install -y -qq tesseract-ocr tesseract-ocr-por poppler-utils
pip -q install pytesseract pillow


Selecting previously unselected package poppler-utils.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package tesseract-ocr-por.
Preparing to unpack .../tesseract-ocr-por_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
from pathlib import Path
import pytesseract
from PIL import Image
import subprocess

repo = Path("/content/cardioia-fase1-tabagismo-vape")
in_dir = repo / "docs" / "textos"
out_dir = repo / "docs" / "textos_txt"
out_dir.mkdir(parents=True, exist_ok=True)

pdfs = sorted(in_dir.glob("*.pdf"))
print("PDFs:", [p.name for p in pdfs])

def ocr_pdf_to_text(pdf_path: Path) -> str:
    # Converte cada página do PDF em PNGs temporários
    tmp_dir = Path("/content/_tmp_ocr") / pdf_path.stem
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # pdftoppm gera arquivos: page-1.png, page-2.png...
    subprocess.run(
        ["pdftoppm", "-png", "-r", "200", str(pdf_path), str(tmp_dir / "page")],
        check=True
    )

    pages = sorted(tmp_dir.glob("page*.png"))
    texts = []
    for img_path in pages:
        img = Image.open(img_path)
        t = pytesseract.image_to_string(img, lang="por")
        texts.append(t)

    return "\n\n".join(texts).strip()

for pdf in pdfs:
    print("\nOCR:", pdf.name)
    txt = ocr_pdf_to_text(pdf)
    out_path = out_dir / (pdf.stem.replace(" ", "_") + ".txt")
    out_path.write_text(txt, encoding="utf-8")
    print("OK ->", out_path.name, "| chars:", len(txt))


PDFs: ['CARDIO IA - FLAVIA BOCCHINO.pdf', 'bse-3-2-3.pdf', 'caderno_atencao_primaria_29_rastreamento.pdf', 'cartilha1-parar-de-fumar.pdf']

OCR: CARDIO IA - FLAVIA BOCCHINO.pdf
OK -> CARDIO_IA_-_FLAVIA_BOCCHINO.txt | chars: 3893

OCR: bse-3-2-3.pdf
OK -> bse-3-2-3.txt | chars: 26894

OCR: caderno_atencao_primaria_29_rastreamento.pdf
OK -> caderno_atencao_primaria_29_rastreamento.txt | chars: 199762

OCR: cartilha1-parar-de-fumar.pdf
OK -> cartilha1-parar-de-fumar.txt | chars: 39591


In [6]:
%%bash
cd /content/cardioia-fase1-tabagismo-vape
ls -lh docs/textos_txt
echo "---- prévia ----"
for f in docs/textos_txt/*.txt; do
  echo "### $f"
  head -n 15 "$f"
  echo
done


total 276K
-rw-r--r-- 1 root root  27K Feb 12 12:59 bse-3-2-3.txt
-rw-r--r-- 1 root root 202K Feb 12 13:08 caderno_atencao_primaria_29_rastreamento.txt
-rw-r--r-- 1 root root 4.0K Feb 12 12:58 CARDIO_IA_-_FLAVIA_BOCCHINO.txt
-rw-r--r-- 1 root root  40K Feb 12 13:10 cartilha1-parar-de-fumar.txt
---- prévia ----
### docs/textos_txt/bse-3-2-3.txt
Biomedical Science and Engineering, 2015, Vol. 3, No. 2, 41-45
Available online at http://pubs.sciepub.com/bse/3/2/3

O Science and Education Publishing

DOI:10.12691/bse-3-2-3

+ SeltP
vw) Science & Education
Publishing

 

AT1 Receptor Antagonists: Pharmacological Treatment


### docs/textos_txt/caderno_atencao_primaria_29_rastreamento.txt
M NESTÉRD DA SAÚDE

ISBN 978-85-334-1729:

II Ep
Mo ATENÇÃO PRIMÁRIA

Rastream ento

Rastream ento

z

<
=

### docs/textos_txt/CARDIO_IA_-_FLAVIA_BOCCHINO.txt
Flavia Nunes Bocchino RM — 564213
12 DE Fevereiro de 2026

Tabagismo e Vapes como Eixo de Risco Cardiovascular: Base Teórica do
CardiolA (18- 40 anos)